# Exercise 1: Fertility audit

*Neural Morphology · version 0.1.0*

> Take a Turkish news article of at least 500 words and tokenise it with `dbmdz/bert-base-turkish-cased` (32K), `dbmdz/bert-base-turkish-128k-cased` (128K) and `xlm-roberta-base`, without special tokens. Compute fertility twice: words tokenised one at a time, and running text (tokens of the whole text divided by its words). State how you count words and punctuation. What proportion of words are single tokens? What is the 95th-percentile tokens per word?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" "transformers==5.3.0" sentencepiece protobuf scikit-learn stanza requests` (the compiled `hfst` package needs Python 3.11 or 3.12). Tokenisers are loaded from Hugging Face, or from a local copy in TurkicNLP's model folder (`~/.turkicnlp/models/huggingface/<org>--<name>`). Universal Dependencies files are read from the UD GitHub repositories (master branch), so internet access is needed. Cells whose model cannot be loaded print the reason instead of a result.

**Counting.** Words are the Unicode alphanumeric runs of Scripts and Encoding (`[^\W_]+`), so punctuation is not a word. *Isolated* fertility tokenises each word on its own; *running* fertility tokenises the whole text, so punctuation tokens and the tokeniser's own pre-splitting are included in the numerator but not in the denominator. The text is the start of the Turkish X-WMT test set (news), cut at the first sentence that reaches 500 words. Besides the three tokenisers of the exercise, the cell adds Glot500 (a Unigram model that extends the XLM-R vocabulary) and NLLB-200 (a SentencePiece BPE model).

In [ ]:
import os, re, collections, statistics
from pathlib import Path
import requests

UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/master/{file}"
WORD = re.compile(r"[^\W_]+")                         # the book's word definition (Scripts and Encoding)

def ud_sentences(repo, file):
    """CoNLL-U sentences as lists of (form, lemma, upos, feats) for integer-ID words, with the sent_id."""
    text = requests.get(UD.format(repo=repo, file=file), timeout=120)
    text.raise_for_status()
    sents, words, sid = [], [], None
    for line in text.text.splitlines():
        if line.startswith("# sent_id"):
            sid = line.split("=", 1)[1].strip()
        elif not line.strip():
            if words:
                sents.append((sid, words))
            words = []
        elif not line.startswith("#"):
            c = line.split("\t")
            if c[0].isdigit():
                words.append((c[1], c[2], c[3], c[5]))
    if words:
        sents.append((sid, words))
    return sents

def load_tokenizer(name):
    """A Hugging Face tokenizer, from TurkicNLP's local model folder if present, otherwise from the Hub."""
    from transformers import AutoTokenizer
    local = Path("~/.turkicnlp/models/huggingface").expanduser() / name.replace("/", "--")
    return AutoTokenizer.from_pretrained(str(local) if (local / "tokenizer_config.json").exists() else name)

def pieces(tok, word):
    """Subword pieces of one word without special tokens and without the word-initial marker."""
    out = []
    for p in tok.tokenize(word):
        p = p.replace("▁", "").replace("Ġ", "")
        out.append(p[2:] if p.startswith("##") else p)
    return [p for p in out if p]

CORE_FEATS = ("Case", "Number", "Number[psor]", "Person", "Person[psor]", "Tense", "VerbForm", "Voice")
DEFAULTS = {("Case", "Nom"), ("Number", "Sing"), ("Person", "3")}

def core(upos, feats):
    """UPOS plus the core features, without the unmarked defaults Case=Nom, Number=Sing and Person=3,
    and with Polarity only if negative. Treebanks differ in whether they write the defaults out."""
    kv = dict(f.split("=", 1) for f in (feats or "").split("|") if "=" in f)
    keep = {k: v for k, v in kv.items() if k in CORE_FEATS and (k, v) not in DEFAULTS}
    if kv.get("Polarity") == "Neg":
        keep["Polarity"] = "Neg"
    return upos + "|" + "|".join(f"{k}={v}" for k, v in sorted(keep.items()))

In [ ]:
# A Turkish news text: the first X-WMT test sentences until 500 words are reached.
lines = requests.get("https://raw.githubusercontent.com/turkic-interlingua/til-mt/master/xwmt/test/en-tr/en-tr.tr", timeout=60).text.splitlines()
text, n = [], 0
for line in lines:
    text.append(line.strip()); n += len(WORD.findall(line))
    if n >= 500:
        break
text = " ".join(text)
words = WORD.findall(text)
print(f"{len(text.split())} whitespace tokens, {len(words)} words (alphanumeric runs), {len(text)} characters")

TOKENIZERS = {"BERTurk 32K": "dbmdz/bert-base-turkish-cased", "BERTurk 128K": "dbmdz/bert-base-turkish-128k-cased",
              "XLM-R base": "xlm-roberta-base", "Glot500 base": "cis-lmu/glot500-base",
              "NLLB-200": "facebook/nllb-200-distilled-600M"}
print(f"{'tokeniser':14} {'isolated':>9} {'running':>8} {'single':>7} {'p95':>5}")
for label, name in TOKENIZERS.items():
    try:
        tok = load_tokenizer(name)
    except Exception as exc:
        print(f"{label:14} not available here ({type(exc).__name__})")
        continue
    counts = [len(tok.tokenize(w)) for w in words]
    running = len(tok.tokenize(text)) / len(words)
    p95 = statistics.quantiles(counts, n=20)[-1]
    single = sum(c == 1 for c in counts) / len(counts)
    print(f"{label:14} {statistics.mean(counts):9.2f} {running:8.2f} {single:7.1%} {p95:5.1f}")

## Worked solution

Output of `experiments/ch07_exercises/run_ch07_exercises.py` on 19 October 2026:

```
623 whitespace tokens, 516 words (alphanumeric runs), 4069 characters
tokeniser       isolated  running  single   p95
BERTurk 32K    not available here (OSError)
BERTurk 128K   not available here (OSError)
XLM-R base     not available here (OSError)
Glot500 base        1.56     1.91   58.1%   3.0
NLLB-200            1.75     2.11   50.8%   3.0
```

The text has 516 words (623 whitespace tokens; punctuation attached to words is split off by the word definition). The three tokenisers of the exercise could not be loaded in the environment used for the worked solution, because the Hugging Face Hub was not reachable; the cell prints that instead of numbers.

For the two tokenisers available locally:

- **Glot500** needs 1.56 tokens per word when words are tokenised one at a time and 1.91 on running text; 58.1% of the words are a single token, and the 95th percentile is 3 tokens per word.
- **NLLB-200** needs 1.75 and 2.11; 50.8% of the words are a single token, and the 95th percentile is also 3.

Running-text fertility is higher than isolated-word fertility for both, mainly because punctuation becomes tokens of its own but is not counted as words. Report which definition a fertility figure uses: the chapter's fertility table uses isolated words, and its numbers are therefore not directly comparable with running-text counts. These two values are lower than the chapter's figures for Turkish (1.95–2.33) because they come from different tokenisers and a different text (news, not the short challenge sentences).

## Your turn — extension

1. Run the cell where the Hugging Face Hub is reachable, so that the three tokenisers of the exercise are measured, and compare the 32K and 128K BERTurk vocabularies.
2. Repeat with the Kazakh and Uyghur sides of X-WMT and compare with the chapter's fertility table.